#  Gold Layer - Vendor Dimension

The Vendor Dimension contains information about the taxi technology provider responsible for processing each trip.

Instead of storing vendor descriptions repeatedly in the fact table, we maintain a separate dimension table and reference it using a vendor key.

This follows star schema design principles by separating descriptive attributes from transactional data.

### Source

Silver Layer (`taxi.silver.yellow_taxi`)

### Target

`taxi.gold.dim_vendor`

In [0]:
from pyspark.sql import functions as F

silver_df = spark.table("taxi.silver.yellow_taxi")

In [0]:
dim_vendor = (
    silver_df
    .select("VendorID")
    .distinct()
    .withColumnRenamed("VendorID", "vendor_key")
)

In [0]:
dim_vendor = (
    dim_vendor
    .withColumn(
        "vendor_name",
        F.when(
            F.col("vendor_key") == 1,
            "Creative Mobile Technologies"
        )
        .when(
            F.col("vendor_key") == 2,
            "VeriFone Inc."
        )
        .otherwise("Unknown Vendor")
    )
)

In [0]:
display(
    dim_vendor.orderBy("vendor_key")
)

In [0]:
(
    dim_vendor.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("taxi.gold.dim_vendor")
)

In [0]:
display(
    spark.table("taxi.gold.dim_vendor")
)

In [0]:
%sql
DESCRIBE DETAIL taxi.gold.dim_vendor;